# Plots without hexagons

In [2]:
import os
import re
import pandas as pd
import matplotlib.pyplot as plt

directory = "output_lda_boost"
pattern = re.compile(r"^summary_means_stds_[^_]+_[^_]+\.csv$")

# --- gather filenames ---
filenames = [
    fn for fn in os.listdir(directory)
    if pattern.match(fn)
]

# --- 1) Compute per‐target y‐ranges ---
y_ranges = {
    "binary":     {"min": float("inf"), "max": float("-inf")},
    "multiclass": {"min": float("inf"), "max": float("-inf")},
    "three class": {"min": float("inf"), "max": float("-inf")},
}

for fn in filenames:
    target = fn.removeprefix("summary_means_stds_").removesuffix(".csv").split("_")[0]
    df = pd.read_csv(os.path.join(directory, fn))
    df["y_low"]  = df["mean_acc"] - df["std_acc"]
    df["y_high"] = df["mean_acc"] + df["std_acc"]

    y_ranges[target]["min"] = min(y_ranges[target]["min"], df["y_low"].min())
    y_ranges[target]["max"] = max(y_ranges[target]["max"], df["y_high"].max())

# --- 1b) Add a little offset (5% of the span) ---
margin_frac = 0.05
for target, vals in y_ranges.items():
    span = vals["max"] - vals["min"]
    margin = span * margin_frac
    vals["min_offset"] = vals["min"] - margin
    vals["max_offset"] = vals["max"] + margin

# --- 2) Plot with target‐specific y‐limits and offsets ---
color_map  = {0.0: "tab:blue", 0.3: "tab:orange", 0.7: "tab:green"}
marker_map = {0.0: "o",         0.3: "s",          0.7: "^"}

for fn in filenames:
    target, feat_str = fn.removeprefix("summary_means_stds_").removesuffix(".csv").split("_")
    csv_path = os.path.join(directory, fn)
    df = pd.read_csv(csv_path).sort_values(["correlation", "n_features"])

    plt.figure(figsize=(8, 6))
    for corr, grp in df.groupby("correlation"):
        grp = grp.sort_values("n_features")
        plt.errorbar(
            grp["n_features"],
            grp["mean_acc"],
            yerr=grp["std_acc"],
            marker=marker_map.get(corr, "o"),
            capsize=5,
            linewidth=2,
            elinewidth=1.5,
            label=f"\u03C1 ≈ {corr:.1f}",
            color=color_map.get(corr),
        )

    plt.title(
        f"LdaBoost – Accuracy vs {feat_str.removesuffix('k')},000 observations\n"
        f"for a {target} target"
    )
    plt.xlabel("Number of features")
    plt.ylabel("Accuracy")

    # apply the precomputed offset limits
    y_min = y_ranges[target]["min_offset"]
    y_max = y_ranges[target]["max_offset"]
    plt.ylim(0.15, 1)

    plt.grid(True, alpha=0.3)
    plt.legend(frameon=True)
    plt.tight_layout()

    out_fn = f"img_without_bayes/ldaboost_accuracy_{target}_{feat_str}.pdf"
    plt.savefig(out_fn, dpi=300, bbox_inches="tight")
    plt.close()

# Bayes confront

In [3]:
import os
import re
import pandas as pd
import matplotlib.pyplot as plt
from collections import defaultdict

# --- config ---
directory = "output_lda_boost"
bayes_csv = "output_bayes_optimal/bayes_optimal_summary.csv"
pattern = re.compile(r"^summary_means_stds_[^_]+_[^_]+\.csv$")  # summary_means_stds_<target>_<featstr>.csv

# --- gather filenames ---
filenames = [fn for fn in os.listdir(directory) if pattern.match(fn)]

# --- load Bayes CSV robustly ---
bayes_df_raw = pd.read_csv(bayes_csv, header=0)

expected_cols = {"n_samples","n_classes","correlation","n_features","max_acc"}
if not expected_cols.issubset(set(bayes_df_raw.columns)):
    # Fallback: assume columns by position if headers aren't standard
    # Try to map commonly used positions to the fields we need
    tmp = bayes_df_raw.copy()
    tmp_cols = list(tmp.columns)
    # If there are exactly 9 columns like your example, map indices:
    # 0:n_samples, 1:n_classes, 2:correlation, 3:n_features, 8:max_acc
    colmap = {}
    if len(tmp_cols) >= 4:
        colmap[tmp_cols[0]] = "n_samples"
        colmap[tmp_cols[1]] = "n_classes"
        colmap[tmp_cols[2]] = "correlation"
        colmap[tmp_cols[3]] = "n_features"
    # try to find max_acc as last column
    colmap[tmp_cols[-1]] = "max_acc"
    bayes_df = tmp.rename(columns=colmap)
else:
    bayes_df = bayes_df_raw.copy()

# Keep only needed fields and enforce dtypes + normalization
need = ["n_samples","n_classes","correlation","n_features","max_acc"]
bayes_df = bayes_df[[c for c in need if c in bayes_df.columns]].copy()
bayes_df["n_samples"]   = bayes_df["n_samples"].astype(int)
bayes_df["n_classes"]   = bayes_df["n_classes"].astype(int)
bayes_df["correlation"] = bayes_df["correlation"].astype(float).round(3)
bayes_df["n_features"]  = bayes_df["n_features"].astype(int)
bayes_df["max_acc"]     = bayes_df["max_acc"].astype(float)

# --- helpers ---
def parse_samples_hint(feat_str: str):
    s = feat_str.strip().lower()
    try:
        if s.endswith("k"):
            val = float(s[:-1]) * 1000
        else:
            val = float(s)
        n = int(round(val))
    except ValueError:
        return None
    return n if n in set(bayes_df["n_samples"].unique()) else None

_WORD_NUMS = {
    "two": 2, "binary": 2,
    "three": 3, "tri": 3, "ternary": 3,
    "four": 4, "quad": 4,
    "five": 5, "penta": 5,
}

def infer_n_classes_from_token(token: str) -> int | None:
    t = token.strip().lower().replace("-", " ")
    m = re.search(r"\b(\d+)\b", t)
    if m:
        return int(m.group(1))
    for k, v in _WORD_NUMS.items():
        if k in t:
            return v
    if "multi" in t:
        return 5
    return None

def canonicalize_for_bucket(token: str) -> str:
    t = token.strip().lower()
    if "bin" in t or "binary" in t or re.search(r"\b2\b", t):
        return "binary"
    return "multiclass"

def _norm_key(corr, n_feat):
    return (round(float(corr), 3), int(n_feat))

def build_bayes_lookup(n_classes: int | None, n_samples: int | None):
    subset = bayes_df.copy()
    if n_classes is not None:
        subset = subset[subset["n_classes"] == n_classes]
    if n_samples is not None:
        subset = subset[subset["n_samples"] == n_samples]
    return {
        _norm_key(r.correlation, r.n_features): float(r.max_acc)
        for r in subset.itertuples(index=False)
    }

# --- 1) y-range collection across targets (including Bayes max) ---
y_ranges = defaultdict(lambda: {"min": float("inf"), "max": float("-inf")})

for fn in filenames:
    raw_target = fn.removeprefix("summary_means_stds_").removesuffix(".csv").split("_")[0]
    bucket = canonicalize_for_bucket(raw_target)
    df = pd.read_csv(os.path.join(directory, fn))
    df["y_low"]  = df["mean_acc"] - df["std_acc"]
    df["y_high"] = df["mean_acc"] + df["std_acc"]

    y_ranges[bucket]["min"] = min(y_ranges[bucket]["min"], df["y_low"].min())
    y_ranges[bucket]["max"] = max(y_ranges[bucket]["max"], df["y_high"].max())

    _, feat_str = fn.removeprefix("summary_means_stds_").removesuffix(".csv").split("_")
    n_classes = infer_n_classes_from_token(raw_target)
    n_samples_hint = parse_samples_hint(feat_str)

    bsub = bayes_df.copy()
    if n_classes is not None:
        bsub = bsub[bsub["n_classes"] == n_classes]
    if n_samples_hint is not None:
        bsub = bsub[bsub["n_samples"] == n_samples_hint]
    if not bsub.empty:
        y_ranges[bucket]["min"] = min(y_ranges[bucket]["min"], bsub["max_acc"].min())
        y_ranges[bucket]["max"] = max(y_ranges[bucket]["max"], bsub["max_acc"].max())

margin_frac = 0.05
for bucket, vals in y_ranges.items():
    span = vals["max"] - vals["min"]
    margin = span * margin_frac if span > 0 else 0.02
    vals["min_offset"] = vals["min"] - margin
    vals["max_offset"] = vals["max"] + margin

# --- 2) plotting ---
color_map  = {0.0: "tab:blue", 0.3: "tab:orange", 0.7: "tab:green"}
marker_map = {0.0: "o",         0.3: "s",          0.7: "^"}

for fn in filenames:
    raw_target, feat_str = fn.removeprefix("summary_means_stds_").removesuffix(".csv").split("_")
    bucket = canonicalize_for_bucket(raw_target)
    csv_path = os.path.join(directory, fn)
    df = pd.read_csv(csv_path).sort_values(["correlation", "n_features"])
    df["correlation"] = df["correlation"].astype(float).round(3)
    df["n_features"]  = df["n_features"].astype(int)

    n_classes = infer_n_classes_from_token(raw_target)
    n_samples_hint = parse_samples_hint(feat_str)
    bayes_lookup = build_bayes_lookup(n_classes, n_samples_hint)

    plt.figure(figsize=(8, 6))
    total_matches = 0
    for corr, grp in df.groupby("correlation"):
        grp = grp.sort_values("n_features")
        plt.errorbar(
            grp["n_features"],
            grp["mean_acc"],
            yerr=grp["std_acc"],
            marker=marker_map.get(corr, "o"),
            capsize=5,
            linewidth=2,
            elinewidth=1.5,
            label=f"\u03C1 ≈ {corr:.1f}",
            color=color_map.get(corr),
        )

        xs, ys = [], []
        for x in grp["n_features"]:
            key = _norm_key(corr, x)
            if key in bayes_lookup:
                xs.append(x)
                ys.append(bayes_lookup[key])

        total_matches += len(xs)
        if xs:
            plt.scatter(
                xs, ys,
                marker="h",
                s=110,
                edgecolor="black",
                linewidths=0.8,
                zorder=5,
                label=None,
                color=color_map.get(corr),
            )

    print(f"[INFO] {fn}: overlaid {total_matches} Bayes hexagon(s).")

    # Keep your title style; show the raw_target phrase for clarity
    pretty_obs = f"{feat_str.removesuffix('k')},000" if feat_str.endswith("k") else feat_str
    plt.title(
        f"LdaBoost – Accuracy vs {pretty_obs} observations\n"
        f"for a {raw_target} target"
    )
    plt.xlabel("Number of features")
    plt.ylabel("Accuracy")

    y_min = y_ranges[bucket]["min_offset"]
    y_max = y_ranges[bucket]["max_offset"]
    plt.ylim(0.2, 1)

    plt.grid(True, alpha=0.3)
    plt.legend(frameon=True)
    plt.tight_layout()

    out_fn = f"img/ldaboost_accuracy_{raw_target}_{feat_str}.pdf"
    plt.savefig(out_fn, dpi=600, bbox_inches="tight")
    plt.close()

[INFO] summary_means_stds_multiclass_10k.csv: overlaid 15 Bayes hexagon(s).
[INFO] summary_means_stds_binary_2k.csv: overlaid 15 Bayes hexagon(s).
[INFO] summary_means_stds_multiclass_2k.csv: overlaid 15 Bayes hexagon(s).
[INFO] summary_means_stds_three class_1k.csv: overlaid 15 Bayes hexagon(s).
[INFO] summary_means_stds_binary_10k.csv: overlaid 15 Bayes hexagon(s).
[INFO] summary_means_stds_multiclass_1k.csv: overlaid 15 Bayes hexagon(s).
[INFO] summary_means_stds_binary_1k.csv: overlaid 15 Bayes hexagon(s).


In [4]:
import os
import re
import pandas as pd
import matplotlib.pyplot as plt
from collections import defaultdict

# --- Pipeline confront plots ---
# This cell reads every CSV in `output_pipeline_confront/` matching
#   pipeline_confront_summary_<target>_<samples>.csv
# and saves one PDF per CSV in `img_pipeline_confront/`.

in_dir = "output_pipeline_confront"
out_dir = "img_pipeline_confront"
pattern = re.compile(r"^pipeline_confront_summary_([^_]+)_(.+)\.csv$")

# Gather target/sample from filenames
filenames = [fn for fn in os.listdir(in_dir) if pattern.match(fn)]
filenames.sort()

os.makedirs(out_dir, exist_ok=True)

# 1) Compute y-ranges per target across all files, using mean ± std
y_ranges = defaultdict(lambda: {"min": float("inf"), "max": float("-inf")})
for fn in filenames:
    m = pattern.match(fn)
    raw_target, sample_str = m.group(1), m.group(2)
    df = pd.read_csv(os.path.join(in_dir, fn))
    df["y_low"] = df["mean_acc"] - df["std_acc"]
    df["y_high"] = df["mean_acc"] + df["std_acc"]
    y_ranges[raw_target]["min"] = min(y_ranges[raw_target]["min"], df["y_low"].min())
    y_ranges[raw_target]["max"] = max(y_ranges[raw_target]["max"], df["y_high"].max())

# Add small offset margins
margin_frac = 0.05
for target, vals in y_ranges.items():
    span = vals["max"] - vals["min"]
    margin = span * margin_frac if span > 0 else 0.02
    vals["min_offset"] = vals["min"] - margin
    vals["max_offset"] = vals["max"] + margin

# 2) Plotting style (consistent color/marker maps)
pipeline_colors = {"LdaBoost": "tab:blue", "GBM": "tab:orange", "LDA+GBM": "tab:green"}
pipeline_markers = {"LdaBoost": "o", "GBM": "s", "LDA+GBM": "^"}

# Map targets for display/naming consistency
# - ternary -> three class
# - quinary -> multiclass
def map_target_name(raw: str) -> str:
    t = raw.strip().lower()
    if t == "ternary":
        return "three class"
    if t == "quinary":
        return "multiclass"
    return raw

for fn in filenames:
    m = pattern.match(fn)
    raw_target, sample_str = m.group(1), m.group(2)
    target_name = map_target_name(raw_target)
    csv_path = os.path.join(in_dir, fn)

    df = pd.read_csv(csv_path).copy()
    # Ensure expected types and sorting
    if "n_features" in df.columns:
        df["n_features"] = df["n_features"].astype(int)
    df = df.sort_values(["pipeline", "n_features"]) if "pipeline" in df.columns else df

    plt.figure(figsize=(8, 6))

    # One curve per pipeline
    if "pipeline" in df.columns:
        for pipe_name, grp in df.groupby("pipeline"):
            grp = grp.sort_values("n_features")
            plt.errorbar(
                grp["n_features"],
                grp["mean_acc"],
                yerr=grp["std_acc"],
                marker=pipeline_markers.get(pipe_name, "o"),
                capsize=5,
                linewidth=2,
                elinewidth=1.5,
                label=str(pipe_name),
                color=pipeline_colors.get(pipe_name),
            )
    else:
        # Fallback: plot a single series if 'pipeline' column is absent
        grp = df.sort_values("n_features")
        plt.errorbar(
            grp["n_features"],
            grp["mean_acc"],
            yerr=grp["std_acc"],
            marker="o",
            capsize=5,
            linewidth=2,
            elinewidth=1.5,
            label="Accuracy",
            color="tab:blue",
        )

    # Title consistent with earlier style
    pretty_obs = f"{sample_str.removesuffix('k')},000" if sample_str.endswith("k") else sample_str
    plt.title(
        f"Pipeline Confront – Accuracy vs {pretty_obs} observations\n"
        f"for a {target_name} target"
    )
    plt.xlabel("Number of features")
    plt.ylabel("Accuracy")

    # Apply target-specific y-limits with a global clamp similar to earlier cells
    y_min = y_ranges[raw_target]["min_offset"]
    y_max = y_ranges[raw_target]["max_offset"]
    plt.ylim(0.2, 1)

    plt.grid(True, alpha=0.3)
    plt.legend(frameon=True)
    plt.tight_layout()

    out_fn = os.path.join(out_dir, f"pipeline_confront_accuracy_{target_name}_{sample_str}.pdf")
    plt.savefig(out_fn, dpi=600, bbox_inches="tight")
    plt.close()

print(f"Saved {len(filenames)} PDF plot(s) to '{out_dir}'.")


Saved 3 PDF plot(s) to 'img_pipeline_confront'.
